In [ ]:
de=(2251262576%15)+1
de


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from hmmlearn import hmm
import time
# from scipy.stats import multivariate_normal
from sklearn.metrics import accuracy_score
from numpy.random import choice
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment

In [ ]:
df=pd.read_csv('seattle-weather.csv')

In [ ]:
df.isnull().sum()

In [ ]:
le=LabelEncoder()
df['weather_encoder']=le.fit_transform(df['weather'])


In [ ]:
# L?y c?c c?t quan s?t
observations=df[['precipitation','temp_max','wind']].values
states=df['weather_encoder'].values

In [ ]:
#Chia d? li?u th?nh t?p hu?n luy?n v? ki?m tra
train_size = int(0.8 * len(df))
train_observations = observations[:train_size]
test_observations = observations[train_size:]
train_states = states[:train_size]
test_states = states[train_size:]

In [ ]:
start_time_em=time.time()

In [ ]:
#Kh?i t?o m? h?nh HMM
n_states = len(np.unique(states))
model_em=hmm.GaussianHMM(n_components=n_states,covariance_type='full',n_iter=100,random_state=42)
model_em.fit(train_observations)


In [ ]:
em_preds = model_em.predict(test_observations)
cm = confusion_matrix(test_states, em_preds)

# 2. D?ng thu?t to?n Hungarian ?? t?m ?nh x? t?i ?u
row_ind, col_ind = linear_sum_assignment(-cm)  # d?u tr? v? c?n maximize matching

# 3. T?o ?nh x? t? nh?n EM sang nh?n th?t
mapping = dict(zip(col_ind, row_ind))

# 4. ??i l?i nh?n EM theo ?nh x?
em_preds_aligned = np.array([mapping[p] for p in em_preds])

# 5. T?nh l?i accuracy
em_acc= accuracy_score(test_states, em_preds_aligned)
print("EM Accuracy (aligned):", em_acc)

In [ ]:
# T?nh th?i gian hu?n luy?n
time_em=time.time()-start_time_em

In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix


In [ ]:

class ParticleFilterHMM:
    def __init__(self, n_states, n_particles=100):
        self.n_states = n_states
        self.n_particles = n_particles

    def initialize(self, observations, states):
        # Kh?i t?o x?c su?t ban ??u, ma tr?n chuy?n ti?p v? ph?n ph?i quan s?t
        self.start_prob = np.bincount(states[:10], minlength=self.n_states) / 10
        self.trans_mat = np.zeros((self.n_states, self.n_states))
        for i in range(len(states) - 1):
            self.trans_mat[states[i], states[i + 1]] += 1
        self.trans_mat = (self.trans_mat.T / np.sum(self.trans_mat, axis=1)).T

        self.means = np.zeros((self.n_states, observations.shape[1]))
        self.covs = np.zeros((self.n_states, observations.shape[1], observations.shape[1]))
        for s in range(self.n_states):
            obs_s = observations[states == s]
            self.means[s] = np.mean(obs_s, axis=0)
            self.covs[s] = np.cov(obs_s.T) + np.eye(observations.shape[1]) * 1e-6

    def particle_filter(self, observations):
        T = len(observations)
        particles = choice(self.n_states, size=self.n_particles, p=self.start_prob)
        weights = np.ones(self.n_particles) / self.n_particles
        estimated_states = []

        for t in range(T):
            obs = observations[t]
            # D? ?o?n b??c ti?p theo b?ng ma tr?n chuy?n ti?p
            new_particles = np.array([choice(self.n_states, p=self.trans_mat[s]) for s in particles])
            
            # T?nh tr?ng s? d?a tr?n x?c su?t quan s?t
            for i, s in enumerate(new_particles):
                try:
                    mean = self.means[s]
                    cov = self.covs[s]
                    diff = obs - mean
                    likelihood = np.exp(-0.5 * diff.T @ np.linalg.inv(cov) @ diff)
                    likelihood /= np.sqrt((2 * np.pi)**len(obs) * np.linalg.det(cov))
                except:
                    likelihood = 1e-6
                weights[i] *= likelihood

            # Chu?n h?a tr?ng s?
            weights += 1e-300  # tr?nh chia cho 0
            weights /= np.sum(weights)

            # ??c l??ng tr?ng th?i: mode ho?c expectation
            estimated_state = np.bincount(new_particles, weights=weights).argmax()
            estimated_states.append(estimated_state)

            # Resample
            indices = choice(range(self.n_particles), size=self.n_particles, p=weights)
            particles = new_particles[indices]
            weights = np.ones(self.n_particles) / self.n_particles

        return estimated_states

# ===================
# Hu?n luy?n v? d? ?o?n
# ===================

n_states = len(np.unique(states))
pf_hmm = ParticleFilterHMM(n_states=n_states, n_particles=200)
pf_hmm.initialize(train_observations, train_states)
predicted_states = pf_hmm.particle_filter(test_observations)

# ===================
# ??nh gi?
# ===================
from sklearn.metrics import accuracy_score
pf_acc=accuracy_score(test_states, predicted_states)

print("Accuracy:",pf_acc )

In [ ]:
print("So s?nh ?? ch?nh x?c HMM:")
print(f" HMM v?i EM (Baum-Welch):       {em_acc:.4f}")
print(f" HMM v?i Particle Filter (MC):  {pf_acc:.4f}")

# V? bi?u ?? so s?nh k?t qu?
plt.figure(figsize=(15, 4))
plt.plot(test_states[:200], label='True States')
plt.plot(em_preds[:200], label='EM Prediction', alpha=0.7)
plt.plot(predicted_states[:200], label='Particle Filter Prediction', alpha=0.7)
plt.legend()
plt.title("So s?nh d? ?o?n tr?ng th?i (200 b??c ??u)")
plt.show()